In [ ]:
import tensorflow as tf
import numpy as np
import os
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score
import warnings
from sklearn.metrics import recall_score
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.metrics import confusion_matrix
import seaborn as sns
warnings.simplefilter('ignore')

# ---------------------------
# SETUP
# ---------------------------
os.makedirs("models", exist_ok=True)
dataset_path = Path("/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray")
train_dir = dataset_path / "train"
test_dir  = dataset_path / "test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ---------------------------
# FOCAL LOSS (alpha = 0.6)
# ---------------------------
@tf.keras.utils.register_keras_serializable()
def focal_loss(gamma=2., alpha=0.6):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
        y_true = tf.cast(y_true, tf.float32)

        loss = -y_true * alpha * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
        loss -= (1 - y_true) * (1 - alpha) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)

        return tf.reduce_mean(loss)
    return loss

# ---------------------------
# F1 METRIC
# ---------------------------
def f1_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.round(y_pred)

    tp = tf.reduce_sum(y_true * y_pred)
    precision = tp / (tf.reduce_sum(y_pred) + 1e-7)
    recall = tp / (tf.reduce_sum(y_true) + 1e-7)

    return 2 * (precision * recall) / (precision + recall + 1e-7)

# ---------------------------
# DATASET
# ---------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    color_mode="rgb"
)

# ---------------------------
# AUGMENTATION
# ---------------------------
data_augmentation = tf.keras.Sequential([
   
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
   
])

preprocess_input = tf.keras.applications.efficientnet.preprocess_input

train_ds = train_ds.map(lambda x,y: (preprocess_input(data_augmentation(x)), y)).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
test_ds  = test_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

# ---------------------------
# MODEL (avec BatchNorm)
# ---------------------------
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=(224,224,3),
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.BatchNormalization()(x)   #  ajout
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs, outputs)

# ---------------------------
# COMPILATION
# ---------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    loss=focal_loss(gamma=2., alpha=0.6),
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall(),
        tf.keras.metrics.AUC(),
        f1_metric
    ]
)

model.summary()

# ---------------------------
# CALLBACKS
# ---------------------------
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        mode='min'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        verbose=1,
        mode='min'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "models/best_model_tl.keras",
        save_best_only=True
    )
]

# ---------------------------
# TRAINING
# ---------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks
)

# ---------------------------
# TEST + THRESHOLD OPTIMISÉ (ACCURACY)
# ---------------------------
best_model = tf.keras.models.load_model(
    "models/best_model_tl.keras",
    custom_objects={
        "loss": focal_loss(gamma=2., alpha=0.6),
        "f1_metric": f1_metric
    }
)

y_true = np.concatenate([y for x,y in test_ds], axis=0)
y_pred = best_model.predict(test_ds)
# optimisation par combinaison de deux classes 


best_thresh = 0
best_score = 0

for t in np.arange(0.3, 0.9, 0.01):
    y_pred_labels = (y_pred > t).astype(int)
    
    #  recall pour chaque classe
    recall_normal = recall_score(y_true, y_pred_labels, pos_label=0)
    recall_pneumonia = recall_score(y_true, y_pred_labels, pos_label=1)
    
    #  combinaison équilibrée
    score = 0.4 * recall_normal + 0.6 * recall_pneumonia
    
    if score > best_score:
        best_score = score
        best_thresh = t

print("Best threshold (balanced):", best_thresh)
print("Best balanced score:", best_score)

# ---------------------------
# ÉVALUATION FINALE
# ---------------------------
y_pred_labels = (y_pred > best_thresh).astype(int)

print(classification_report(
    y_true,
    y_pred_labels,
    target_names=["NORMAL", "PNEUMONIA"]
))
# ---------------------------
# MATRICE DE CONFUSION
# ---------------------------
cm = confusion_matrix(y_true, y_pred_labels)
class_names = ["NORMAL", "PNEUMONIA"]

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.5,
    linecolor='white'
)
plt.xlabel("Prédit", fontsize=12)
plt.ylabel("Réel", fontsize=12)
plt.title("Matrice de confusion", fontsize=13)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

# ---------------------------
# COURBE DE LOSS
# ---------------------------
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train',      color='#185FA5', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation', color='#D85A30', linewidth=2, linestyle='--')
plt.xlabel("Époque", fontsize=12)
plt.ylabel("Focal Loss", fontsize=12)
plt.title("Courbe de loss", fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
# ===============================
# IMPORTS
# ===============================
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# ===============================
# METRICS / LOSS (si déjà utilisés)
# ===============================
def f1_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.round(y_pred)

    tp = tf.reduce_sum(y_true * y_pred)
    precision = tp / (tf.reduce_sum(y_pred) + 1e-7)
    recall = tp / (tf.reduce_sum(y_true) + 1e-7)

    return 2 * (precision * recall) / (precision + recall + 1e-7)

def focal_loss(gamma=2., alpha=0.6):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        pt = tf.exp(-bce)
        return alpha * (1 - pt) ** gamma * bce
    return loss

# ===============================
# GRAD-CAM (VERSION STABLE)
# ===============================
def make_gradcam_heatmap(img_array, model):
    
    #  récupérer EfficientNet
    base_model = model.get_layer("efficientnetb0")

    #  trouver automatiquement la dernière couche conv
    for layer in reversed(base_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer
            print("Dernière couche conv utilisée:", layer.name)
            break

    #  modèle conv
    conv_model = tf.keras.Model(
        inputs=base_model.input,
        outputs=last_conv_layer.output
    )

    #  classifier (reste du modèle)
    classifier_input = tf.keras.Input(shape=conv_model.output.shape[1:])
    x = classifier_input

    for layer in model.layers[2:]:  # skip input + efficientnet
        x = layer(x)

    classifier_model = tf.keras.Model(classifier_input, x)

    #  forward + gradients
    with tf.GradientTape() as tape:
        conv_outputs = conv_model(img_array)
        tape.watch(conv_outputs)

        predictions = classifier_model(conv_outputs)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

# ===============================
# OVERLAY
# ===============================
def overlay_heatmap(img, heatmap, alpha=0.6):
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    return cv2.addWeighted(img.astype(np.uint8), 1.0, heatmap, alpha, 0)

# ===============================
# PARAMÈTRES
# ===============================
IMG_SIZE = (224, 224)

# ===============================
# LOAD MODEL
# ===============================
best_model = tf.keras.models.load_model(
    "models/best_model_tl.keras",
    custom_objects={
        "loss": focal_loss(gamma=2., alpha=0.6),
        "f1_metric": f1_metric
    }
)

# ===============================
# LOAD TEST IMAGES (10 images)
# ===============================
test_dir = Path("/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test")
test_images = list(test_dir.rglob("*.jpeg"))[:10]

# ===============================
# LOOP SUR 10 IMAGES
# ===============================
for img_path in test_images:

    # 🔹 load image
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)

    img_array_exp = np.expand_dims(img_array, axis=0)
    img_array_exp = tf.keras.applications.efficientnet.preprocess_input(img_array_exp)

    # 🔹 prediction
    preds = best_model.predict(img_array_exp, verbose=0)
    prob = preds[0][0]
    pred_class = "PNEUMONIA" if prob > 0.5 else "NORMAL"

    # 🔹 Grad-CAM
    heatmap = make_gradcam_heatmap(img_array_exp, best_model)

    # 🔹 overlay
    superimposed_img = overlay_heatmap(np.array(img), heatmap)

    # ===============================
    # AFFICHAGE
    # ===============================
    plt.figure(figsize=(12,4))
    plt.suptitle(f"{img_path.name} | Pred: {pred_class} ({prob:.2f})")

    plt.subplot(1,3,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(heatmap, cmap='jet')
    plt.title("Heatmap")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM")
    plt.axis("off")

    plt.show()

In [ ]:
!pip install owlready2

In [ ]:
# ================================
# ONTOLOGIE + EXPLICATION (1 CELLULE)(version correcte)
# ================================

from owlready2 import *

# -------------------------
# 1. CREATION ONTOLOGIE
# -------------------------
onto = get_ontology("http://pneumo.org/ontology.owl")

with onto:

    # Anatomie
    class Organe(Thing): pass
    class Poumon(Organe): pass
    class PoumonDroit(Poumon): pass
    class PoumonGauche(Poumon): pass

    # Etats pulmonaires
    class EtatPulmonaire(Thing): pass
    class Normal(EtatPulmonaire): pass
    class Pneumonie(EtatPulmonaire): pass

    # Signes radiologiques
    class SigneRadiologique(Thing): pass
    class Opacite(SigneRadiologique): pass
    class Infiltration(SigneRadiologique): pass

    # Relations
    class a_localisation(ObjectProperty):
        domain = [EtatPulmonaire]
        range = [Poumon]

    class a_signe(ObjectProperty):
        domain = [EtatPulmonaire]
        range = [SigneRadiologique]

    class indique(ObjectProperty):
        domain = [SigneRadiologique]
        range = [EtatPulmonaire]

    # Connaissance médicale
    Opacite.indique = [Pneumonie]
    Infiltration.indique = [Pneumonie]

onto.save(file="pneumo_ontology.owl")
# -------------------------
# 2. FONCTION EXPLICATION
# -------------------------
def explain_prediction(pred, confidence, zone="inconnue"):
    
    if pred == 0:
        return f"Aucune anomalie détectée (confiance: {confidence:.2f})."
    
    # mapping zone
    if zone == "gauche":
        loc = "poumon gauche"
    elif zone == "droite":
        loc = "poumon droit"
    else:
        loc = "zone pulmonaire non précisée"

    # créer instance ontologie
    with onto:
        patient = onto.Pneumonie(f"patient_{np.random.randint(1000)}")
        
        if zone == "gauche":
            patient.a_localisation = [onto.PoumonGauche()]
        elif zone == "droite":
            patient.a_localisation = [onto.PoumonDroit()]
        
        patient.a_signe = [onto.Opacite()]

    # texte final
    return f"Anomalie détectée dans le {loc}, caractérisée par une opacité pulmonaire, suggérant une pneumonie (confiance: {confidence:.2f})."

# -------------------------
# 3. APPLICATION SUR TES RESULTATS
# -------------------------

print("\n===== EXPLICATIONS XAI + ONTOLOGIE =====\n")

indices = np.random.choice(len(y_pred_labels), size=50, replace=False)

for i in indices:
    
    pred = int(y_pred_labels[i][0])
    confidence = float(y_pred[i][0])
    
    #  ici on peut remplacer par Grad-CAM plus tard
    zone = np.random.choice(["gauche", "droite"])
    
    explanation = explain_prediction(pred, confidence, zone)
    
    print(f"Image {i+1}")
    print("Prediction:", "PNEUMONIA" if pred == 1 else "NORMAL")
    print("Explanation:", explanation)
    print("-"*60)

In [ ]:
# ===============================
# COMBINAISON GRAD-CAM + ONTOLOGIE
# ===============================

from owlready2 import *
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf

# -------------------------
# FONCTION POUR DÉTECTER LA ZONE DOMINANTE
# -------------------------
def detect_zone(heatmap):
    """
    Détecte si la zone chaude se situe plutôt à gauche ou à droite
    de l'image. Renvoie "gauche", "droite" ou "inconnue".
    """
    h, w = heatmap.shape
    left_sum = np.sum(heatmap[:, :w//2])
    right_sum = np.sum(heatmap[:, w//2:])
    
    if left_sum > right_sum * 1.1:
        return "gauche"
    elif right_sum > left_sum * 1.1:
        return "droite"
    else:
        return "inconnue"

# -------------------------
# BOUCLE SUR LES IMAGES
# -------------------------
for img_path in test_images:

    # 🔹 load image
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array_exp = np.expand_dims(img_array, axis=0)
    img_array_exp = tf.keras.applications.efficientnet.preprocess_input(img_array_exp)

    # 🔹 prediction
    preds = best_model.predict(img_array_exp, verbose=0)
    prob = preds[0][0]
    pred_class = "PNEUMONIA" if prob > 0.5 else "NORMAL"

    # 🔹 Grad-CAM
    heatmap = make_gradcam_heatmap(img_array_exp, best_model)

    # 🔹 overlay
    superimposed_img = overlay_heatmap(np.array(img), heatmap)

    # 🔹 détecter la zone dominante
    zone = detect_zone(heatmap)

    # 🔹 explication + ontologie
    explanation = explain_prediction(int(prob>0.5), float(prob), zone)

    # ===============================
    # AFFICHAGE
    # ===============================
    plt.figure(figsize=(12,4))
    plt.suptitle(f"{img_path.name} | Pred: {pred_class} ({prob:.2f}) | Zone: {zone}")

    plt.subplot(1,3,1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(heatmap, cmap='jet')
    plt.title("Heatmap")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM + Ontologie")
    plt.axis("off")

    plt.show()

    print("Explication XAI + Ontologie :")
    print(explanation)
    print("-"*60)

In [ ]:
!pip install gradio

In [ ]:
# ============================================================
# GRADIO DEMO v2 — Pneumonia Detection + Grad-CAM + Ontologie
# Ontologie en paragraphe (pas de tableau)
# ============================================================

import subprocess
subprocess.run(["pip", "install", "gradio", "owlready2", "--quiet"], check=True)

import os, io, warnings
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from owlready2 import *
import gradio as gr

warnings.simplefilter("ignore")

# ============================================================
# 1. FOCAL LOSS + F1
# ============================================================
@tf.keras.utils.register_keras_serializable()
def focal_loss(gamma=2., alpha=0.6):
    def loss(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred  = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
        y_true  = tf.cast(y_true, tf.float32)
        l  = -y_true * alpha * tf.pow(1 - y_pred, gamma) * tf.math.log(y_pred)
        l -= (1 - y_true) * (1 - alpha) * tf.pow(y_pred, gamma) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(l)
    return loss

def f1_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.round(y_pred)
    tp        = tf.reduce_sum(y_true * y_pred)
    precision = tp / (tf.reduce_sum(y_pred) + 1e-7)
    recall    = tp / (tf.reduce_sum(y_true) + 1e-7)
    return 2 * (precision * recall) / (precision + recall + 1e-7)

# ============================================================
# 2. CHARGEMENT DU MODÈLE
# ============================================================
MODEL_PATH = "models/best_model_tl.keras"
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"loss": focal_loss(gamma=2., alpha=0.6), "f1_metric": f1_metric}
)
print(" Modèle chargé.")

# ============================================================
# 3. ONTOLOGIE
# ============================================================
onto = get_ontology("http://pneumo.org/ontology.owl")
with onto:
    class Organe(Thing): pass
    class Poumon(Organe): pass
    class PoumonDroit(Poumon): pass
    class PoumonGauche(Poumon): pass
    class EtatPulmonaire(Thing): pass
    class Normal(EtatPulmonaire): pass
    class Pneumonie(EtatPulmonaire): pass
    class SigneRadiologique(Thing): pass
    class Opacite(SigneRadiologique): pass
    class Infiltration(SigneRadiologique): pass
    class a_localisation(ObjectProperty):
        domain = [EtatPulmonaire]; range = [Poumon]
    class a_signe(ObjectProperty):
        domain = [EtatPulmonaire]; range = [SigneRadiologique]
    class indique(ObjectProperty):
        domain = [SigneRadiologique]; range = [EtatPulmonaire]
    Opacite.indique      = [Pneumonie]
    Infiltration.indique = [Pneumonie]

# ============================================================
# 4. GRAD-CAM
# ============================================================
IMG_SIZE = (224, 224)

def make_gradcam_heatmap(img_array, model):
    base_model = model.get_layer("efficientnetb0")
    last_conv  = None
    for layer in reversed(base_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv = layer; break
    conv_model = tf.keras.Model(inputs=base_model.input, outputs=last_conv.output)
    clf_input  = tf.keras.Input(shape=conv_model.output.shape[1:])
    x = clf_input
    for layer in model.layers[2:]:
        x = layer(x)
    clf_model = tf.keras.Model(clf_input, x)
    with tf.GradientTape() as tape:
        conv_out = conv_model(img_array)
        tape.watch(conv_out)
        score    = clf_model(conv_out)[:, 0]
    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(img, heatmap, alpha=0.55):
    h_uint8   = np.uint8(255 * heatmap)
    h_resized = cv2.resize(h_uint8, (img.shape[1], img.shape[0]))
    colored   = cv2.applyColorMap(h_resized, cv2.COLORMAP_JET)
    return cv2.addWeighted(img.astype(np.uint8), 1.0, colored, alpha, 0)

def detect_zone(heatmap):
    h, w      = heatmap.shape
    left_sum  = np.sum(heatmap[:, :w//2])
    right_sum = np.sum(heatmap[:, w//2:])
    if   left_sum  > right_sum * 1.1: return "gauche"
    elif right_sum > left_sum  * 1.1: return "droite"
    else:                              return "bilatérale"

# ============================================================
# 5. EXPLICATION ONTOLOGIQUE → paragraphe narratif
# ============================================================
_cpt = 0

def explain_prediction_html(pred, prob, zone):
    global _cpt; _cpt += 1

    loc_map   = {"gauche": "poumon gauche", "droite": "poumon droit",
                 "bilatérale": "les deux poumons"}
    loc_label = loc_map.get(zone, "zone non précisée")

    if pred == 0:
        conf   = (1 - prob) * 100
        badge  = '<span style="background:#1a4731;color:#3fb950;padding:4px 12px;border-radius:20px;font-weight:700;font-size:1rem;">✅ NORMAL</span>'
        border = "#238636"
        paragraphe = (
            f"L'analyse de cette radiographie thoracique n'a révélé aucune anomalie significative "
            f"(confiance : <b style='color:#3fb950;'>{conf:.1f}%</b>). "
            f"Selon notre base de connaissances ontologique, l'état pulmonaire est classé comme "
            f"<b>Normal</b>, appartenant à la classe <code>EtatPulmonaire</code>. "
            f"Aucun signe radiologique anormal n'a été identifié, et aucune relation "
            f"<code>a_signe</code> vers une <code>SigneRadiologique</code> pathologique "
            f"n'a été instanciée. La radiographie est dans les limites de la normale."
        )
        owl_chain = (
            "<code>Normal</code> ∈ <code>EtatPulmonaire</code> — "
            "aucun signe radiologique → aucune pneumonie inférée."
        )

    else:
        conf   = prob * 100
        badge  = '<span style="background:#4d1519;color:#f85149;padding:4px 12px;border-radius:20px;font-weight:700;font-size:1rem;">🔴 PNEUMONIE</span>'
        border = "#da3633"

        with onto:
            p = onto.Pneumonie(f"patient_{_cpt}")
            if zone == "gauche":   p.a_localisation = [onto.PoumonGauche()]
            elif zone == "droite": p.a_localisation = [onto.PoumonDroit()]
            p.a_signe = [onto.Opacite()]

        loc_onto = {"gauche": "<code>PoumonGauche</code>",
                    "droite": "<code>PoumonDroit</code>",
                    "bilatérale": "<code>PoumonGauche</code> et <code>PoumonDroit</code>"}
        loc_code = loc_onto.get(zone, "<code>Poumon</code>")

        paragraphe = (
            f"L'analyse de cette radiographie thoracique met en évidence une anomalie pulmonaire "
            f"localisée dans le <b>{loc_label}</b> (confiance : <b style='color:#f85149;'>{conf:.1f}%</b>). "
            f"La carte d'activation Grad-CAM confirme que la zone chaude est concentrée à {zone}. "
            f"Selon notre ontologie OWL, un individu de classe <b>Pneumonie</b> a été instancié "
            f"avec une relation <code>a_localisation</code> vers {loc_code} et une relation "
            f"<code>a_signe</code> vers <code>Opacité</code>. "
            f"Par l'axiome <code>Opacité → indique → Pneumonie</code>, le raisonneur ontologique "
            f"confirme la présence d'une consolidation pulmonaire compatible avec une pneumonie "
            f"dans le {loc_label}."
        )
        owl_chain = (
            f"<code>Pneumonie</code> ∈ <code>EtatPulmonaire</code> — "
            f"<code>a_localisation</code> → {loc_code} — "
            f"<code>a_signe</code> → <code>Opacité</code> — "
            f"<code>indique</code> → <code>Pneumonie</code>"
        )

    return f"""
    <div style="background:#161b22;border:1px solid {border};border-radius:10px;padding:16px 18px;
                font-family:'Segoe UI',sans-serif;font-size:13px;line-height:1.7;color:#c9d1d9;">

      <!-- Badge + zone -->
      <div style="margin-bottom:12px;">
        {badge}
        <span style="color:#8b949e;margin-left:12px;font-size:12px;">
          Zone activée : <b style="color:#c9d1d9;">{zone}</b>
        </span>
      </div>

      <!-- Titre section -->
      <div style="font-size:11px;font-weight:700;color:#8b949e;letter-spacing:.07em;
                  margin-bottom:8px;">🧬 EXPLICATION ONTOLOGIQUE (OWL)</div>

      <!-- Paragraphe narratif -->
      <p style="margin:0 0 12px 0;color:#e6edf3;line-height:1.75;font-size:13px;">
        {paragraphe}
      </p>

      <!-- Chaîne OWL résumée -->
      <div style="padding:10px 14px;background:#0d1117;border-radius:6px;
                  border-left:3px solid {border};color:#8b949e;font-size:12px;">
        <b style="color:#c9d1d9;">Chaîne d'inférence OWL :</b><br>
        <span style="line-height:2;">{owl_chain}</span>
      </div>

    </div>"""

# ============================================================
# 6. PIPELINE PRINCIPAL
# ============================================================
def analyze_xray(pil_image):
    if pil_image is None:
        return None, "<p style='color:#f85149'> Veuillez charger une image.</p>"

    img_rgb   = pil_image.convert("RGB").resize(IMG_SIZE)
    img_array = np.array(img_rgb)
    img_exp   = np.expand_dims(img_array.astype("float32"), axis=0)
    img_exp   = tf.keras.applications.efficientnet.preprocess_input(img_exp)

    prob  = float(best_model.predict(img_exp, verbose=0)[0][0])
    pred  = int(prob > 0.5)
    label = "PNEUMONIE" if pred == 1 else "NORMAL"
    color = "#f85149"   if pred == 1 else "#3fb950"

    heatmap      = make_gradcam_heatmap(img_exp, best_model)
    superimposed = overlay_heatmap(img_array.copy(), heatmap)
    zone         = detect_zone(heatmap)

    # Figure matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.patch.set_facecolor("#0d1117")
    for ax, im, t, cm in zip(axes,
                              [img_array, heatmap, superimposed],
                              ["Radiographie", "Heatmap", "Grad-CAM"],
                              [None, "jet", None]):
        ax.imshow(im, cmap=cm)
        ax.set_title(t, color="#8b949e", fontsize=10, pad=6)
        ax.axis("off")
    conf_disp = prob if pred == 1 else 1 - prob
    fig.suptitle(f"Prédiction : {label}   |   Confiance : {conf_disp:.1%}   |   Zone : {zone}",
                 color=color, fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout(pad=0.5)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight", facecolor="#0d1117")
    plt.close(fig)
    buf.seek(0)
    result_img = Image.open(buf)

    # Barres de confiance
    pneu_w = f"{prob*100:.1f}%"
    norm_w = f"{(1-prob)*100:.1f}%"
    bars_html = f"""
    <div style="background:#0d1117;border:1px solid #30363d;border-radius:10px;
                padding:14px 18px;font-family:'Segoe UI',sans-serif;font-size:13px;
                color:#c9d1d9;margin-bottom:10px;">
      <div style="font-weight:700;color:#8b949e;margin-bottom:10px;letter-spacing:.05em;">
        PROBABILITÉS DE CLASSE
      </div>
      <div style="margin-bottom:8px;">
        <div style="display:flex;justify-content:space-between;">
          <span>🔴 PNEUMONIE</span><b style="color:#f85149;">{pneu_w}</b>
        </div>
        <div style="background:#21262d;border-radius:5px;height:12px;margin-top:4px;overflow:hidden;">
          <div style="width:{pneu_w};background:linear-gradient(90deg,#b91c1c,#f85149);height:100%;border-radius:5px;"></div>
        </div>
      </div>
      <div>
        <div style="display:flex;justify-content:space-between;">
          <span>🟢 NORMAL</span><b style="color:#3fb950;">{norm_w}</b>
        </div>
        <div style="background:#21262d;border-radius:5px;height:12px;margin-top:4px;overflow:hidden;">
          <div style="width:{norm_w};background:linear-gradient(90deg,#166534,#3fb950);height:100%;border-radius:5px;"></div>
        </div>
      </div>
    </div>"""

    onto_html = explain_prediction_html(pred, prob, zone)
    return result_img, bars_html + onto_html

# ============================================================
# 7. INTERFACE GRADIO
# ============================================================
CSS = """
body, .gradio-container { background:#0d1117 !important; color:#c9d1d9 !important; }
.gr-button-primary {
    background:#238636 !important; border:1px solid #2ea043 !important;
    color:white !important; font-weight:700 !important;
}
.gr-button-primary:hover { background:#2ea043 !important; }
footer { display:none !important; }
"""

with gr.Blocks(css=CSS, title="Pneumonia XAI Demo") as demo:

    gr.HTML("""
    <div style="text-align:center;padding:18px 0 4px;">
      <h1 style="font-size:1.8rem;font-weight:800;color:#f0f6fc;margin:0;">
         Détection de Pneumonie — XAI Demo
      </h1>
      <p style="color:#484f58;font-size:.85rem;margin-top:4px;">
        EfficientNetB0 · Focal Loss · Grad-CAM · Ontologie OWL
      </p>
    </div>""")

    with gr.Row():
        with gr.Column(scale=1, min_width=260):
            img_input = gr.Image(type="pil", label=" Radiographie thoracique", height=240)
            run_btn   = gr.Button("  Analyser", variant="primary")

        with gr.Column(scale=2, min_width=400):
            img_output = gr.Image(
                label=" Résultats visuels — Original | Heatmap | Grad-CAM",
                height=240,
            )

    html_output = gr.HTML(label="Analyse")

    run_btn.click(fn=analyze_xray, inputs=img_input, outputs=[img_output, html_output])

   

demo.launch(share=True, debug=False, quiet=True)